# Agentic AI Lab
## AI Mini-Movie, First AI Agent, and AI-Assisted Python Development

---

## 1. Objective

This laboratory experiment demonstrates fundamental concepts in Generative and Agentic Artificial Intelligence:
- **Gemini API Setup**: Securely authenticating and generating content using Google's modern Gemini API SDK (`google-genai`).
- **Create a Mini-Movie Using AI**: Transforming a raw creative idea into an organized mini-movie production package (Story, Characters, 5 Scenes with durations, and a cinematic Voiceover script).
- **Build Your First AI Agent**: Implementing an autonomous **Study Planner Agent** in pure Python that perceives a goal and invokes custom Python tools (`get_topics`, `calculate_study_hours`) without third-party frameworks.
- **AI-Assisted Python Development**: Using modern AI coding assistants (GitHub Copilot / Cursor) to **generate**, **debug**, and **document** Python code.

---

## 2. Requirements and Technologies

- **Python (3.10+)**: Core programming environment.
- **Jupyter Notebook**: Interactive notebook format for reproducible execution and lab reporting.
- **Google Gemini API (`google-genai`)**: Modern Google GenAI SDK to interact with multimodal Gemini models (`gemini-2.5-flash`).
- **python-dotenv**: For loading credentials securely from a local `.env` file without exposing keys.
- **Cursor / GitHub Copilot**: AI pair programmer demonstrated for code generation, debugging, and documentation.

# 3. Gemini API Setup

### Aim
To securely load the Gemini API key from the local `.env` file, initialize the modern `google.genai` client, and verify connectivity with a baseline test prompt.

### Key Security Concept
API keys should **never** be hardcoded, printed, or committed to version control. We use `python-dotenv` to read `GEMINI_API_KEY` directly from the environment.

In [25]:
import os
from dotenv import load_dotenv

# 1. Load environment variables from local .env file
load_dotenv()

# 2. Retrieve Gemini API key safely
gemini_api_key = os.getenv("GEMINI_API_KEY")

# 3. Safe check: Verify presence without printing the key value
if gemini_api_key and gemini_api_key.strip():
    print("✅ Gemini API key loaded successfully.")
else:
    print("⚠️ Gemini API key not found. Please ensure GEMINI_API_KEY is configured in your .env file.")

✅ Gemini API key loaded successfully.


### Initializing the Gemini Client and Running a Baseline Query

We use the modern Google GenAI SDK (`google-genai`) and select the officially supported `gemini-2.5-flash` model for fast, capable text generation.

*Note: If `google-genai` is not yet installed in your Python environment, install it via:*
```bash
pip install google-genai
```

In [26]:
# Initialize Google GenAI client and test connectivity
client = None
MODEL_NAME = "gemini-2.5-flash"

try:
    from google import genai

    if gemini_api_key and gemini_api_key.strip():
        # Initialize client with the loaded key
        client = genai.Client(api_key=gemini_api_key)
        
        test_prompt = "Explain Artificial Intelligence in 3 simple sentences."
        print(f"Sending test prompt to Gemini ({MODEL_NAME})...\n")
        
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=test_prompt
        )
        
        print("--- Gemini API Test Response ---")
        print(response.text)
    else:
        print("Skipping API call: GEMINI_API_KEY is not available in .env.")

except ImportError:
    print("❌ The 'google-genai' library is not installed in your Python environment.")
    print("To install it, run: pip install google-genai")
except Exception as e:
    print(f"❌ Error communicating with Gemini API: {e}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Sending test prompt to Gemini (gemini-2.5-flash)...

--- Gemini API Test Response ---
1.  Artificial Intelligence (AI) enables machines to perform tasks that typically require human intelligence.
2.  This involves learning from data, recognizing patterns, understanding language, and making decisions.
3.  Its goal is to solve complex problems, automate processes, and assist humans in various aspects of life.


### Observation
The Gemini API client authenticated cleanly and produced a clear, concise definition of Artificial Intelligence in response to our test prompt. All credentials remained confidential.

---

# 4. Create a Mini-Movie Using AI

### Aim
To design an automated pipeline that takes a user's movie idea and uses Gemini to generate:
1. Movie Title, Genre, Characters, and Short Story
2. A structured 5-scene breakdown (Scene number, title, description, and approximate duration)
3. A synchronized cinematic voiceover narration script

### Workflow

```
User Idea ("An engineering student discovers a robot in his college laboratory")
    ↓
Google Gemini
    ↓
Story Concept & Characters
    ↓
5 Structured Scenes (Title, Description, Duration)
    ↓
Cinematic Voiceover Narration
```

### Technical Note on AI Video Generation
> [!NOTE]
> Standard text-generation LLMs generate script text, structured breakdowns, and prompts. Direct end-to-end video synthesis (such as text-to-video generation like OpenAI Sora or Google Veo) requires specialized video diffusion models and heavy cloud rendering pipelines, which are not part of standard text API endpoints. This experiment demonstrates the fundamental **AI pre-production & scriptwriting pipeline** used by filmmakers.

In [27]:
import json

# Define the user's movie premise
movie_idea = "An engineering student discovers a forgotten sentient robot in his college laboratory during late-night project work."

movie_prompt = f"""
You are a creative filmmaker and screenwriter.
Based on the following movie premise:
"{movie_idea}"

Generate a complete mini-movie plan in valid JSON format.
Your JSON must strictly contain the following keys:
1. "title": A catchy title for the mini-movie.
2. "genre": The film genre (e.g., Sci-Fi, Drama).
3. "characters": A list of main characters, each with a 1-sentence description.
4. "story": A short 2-3 sentence overview of the plot.
5. "scenes": A list of exactly 5 sequential scenes. Each scene must be an object with:
   - "scene_number": Integer (1 to 5)
   - "title": Title of the scene
   - "description": 1-2 sentence visual description of what happens
   - "approximate_duration": e.g., "15 seconds", "20 seconds"

Output ONLY the raw JSON object without markdown code fences if possible.
"""

movie_plan = None

if client:
    try:
        print("🎬 Generating structured mini-movie plan using Gemini...\n")
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=movie_prompt
        )
        
        # Strip markdown fences if present
        raw_text = response.text.strip()
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        if raw_text.startswith("```"):
            raw_text = raw_text[3:]
        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]
        raw_text = raw_text.strip()
        
        movie_plan = json.loads(raw_text)
        
        # Display structured movie details
        print(f"🎬 Title: {movie_plan.get('title')}")
        print(f"🎭 Genre: {movie_plan.get('genre')}")
        print(f"📖 Story: {movie_plan.get('story')}\n")
        
        print("👥 Characters:")
        for char in movie_plan.get("characters", []):
            print(f"  • {char}")
            
        print("\n📽️ 5-Scene Breakdown:")
        for scene in movie_plan.get("scenes", []):
            num = scene.get('scene_number')
            title = scene.get('title')
            dur = scene.get('approximate_duration')
            desc = scene.get('description')
            print(f"  [Scene {num}] {title} ({dur})")
            print(f"    Visual: {desc}")
            
    except Exception as e:
        print(f"Error generating movie plan: {e}")
else:
    print("Skipping mini-movie generation: Gemini client not initialized.")

🎬 Generating structured mini-movie plan using Gemini...

🎬 Title: Project Echo
🎭 Genre: Sci-Fi/Drama
📖 Story: Liam, an exhausted engineering student, stumbles upon a dormant, sentient robot named Echo during a late-night work session. He awakens Echo, uncovering the robot's forgotten history and its longing for connection. Together, they form an unexpected bond, realizing the potential dangers and wonders of Echo's existence within the confines of the college lab.

👥 Characters:
  • {'name': 'Liam', 'description': 'A diligent but stressed engineering student, often found burning the midnight oil in the lab.'}
  • {'name': 'Echo', 'description': "An old, dusty, but remarkably sophisticated sentient robot, forgotten in the lab's archives."}

📽️ 5-Scene Breakdown:
  [Scene 1] The Midnight Grind (20 seconds)
    Visual: Liam, surrounded by schematics and circuit boards, struggles with his project in a deserted, dimly lit college lab. He's clearly frustrated and running on fumes.
  [Scene 2

### Generating the Cinematic Narration / Voiceover Script

Now we take the generated 5 scenes and prompt Gemini to compose a synchronized cinematic voiceover narration for the mini-movie.

In [28]:
if client and movie_plan:
    try:
        scenes_data = json.dumps(movie_plan.get('scenes'), indent=2)
        narration_prompt = f"""
Based on these 5 scenes from our mini-movie '{movie_plan.get('title')}':
{scenes_data}

Write an engaging, cinematic voiceover narration script suitable for a 1-minute mini-movie.
Format clearly scene by scene:
[Scene 1: <Scene Title>] - (Voiceover text)
[Scene 2: <Scene Title>] - (Voiceover text)
... up to Scene 5.
"""
        print("🎙️ Generating cinematic voiceover narration script...\n")
        narration_response = client.models.generate_content(
            model=MODEL_NAME,
            contents=narration_prompt
        )
        
        print("--- Mini-Movie Voiceover Script ---")
        print(narration_response.text)
        
    except Exception as e:
        print(f"Error generating narration script: {e}")
else:
    print("Skipping narration script: movie plan or Gemini client unavailable.")

🎙️ Generating cinematic voiceover narration script...

--- Mini-Movie Voiceover Script ---
Here's an engaging, cinematic voiceover narration script for a 1-minute mini-movie based on your scenes:

**[Scene 1: The Midnight Grind]**
Liam’s world was a blur of schematics and stale coffee. Another deadline loomed, another complex problem refused to yield. The deserted lab echoed with his frustration, the ticking clock the only witness to his struggle.

**[Scene 2: A Glimmer in the Dust]**
Lost in the hunt for a forgotten component, a sudden jolt led to an unexpected discovery. Beneath layers of dust and forgotten dreams, a silent form took shape… ancient, yet oddly captivating.

**[Scene 3: Awakening Echo]**
Intrigue replaced exhaustion. A hidden port, a surge of power, and then… a flicker. A hum. After what felt like an eternity, a dormant spirit began to stir, its optical sensors glowing to life.

**[Scene 4: First Contact]**
Slowly, deliberately, its head turned. The robot's eyes, fille